In [1]:
import requests
from bs4 import BeautifulSoup
import random
import datetime
import re
import lxml

In [2]:
def get_url(url):
    header_list = [
        {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.5790.102 Safari/537.36"},
        {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_3_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.5 Safari/605.1.15"},
        {"User-Agent": "Mozilla/5.0 (Linux; Android 14; Pixel 7 Pro) Gecko/20100101 Firefox/119.0"},
        {"User-Agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Mobile Safari/537.36"}
    ]
    headers = random.choice(header_list)
    baseurl = 'https://www.jumia.com.ng'
    r = requests.get(url, headers=headers, timeout=20)
    soup = BeautifulSoup(r.content, 'lxml')
    links = soup.select('article.prd a')
    return [baseurl + link.attrs['href'] for link in links if link.get('href')]

In [3]:
def get_product(url):
    header_list = [
        {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.5790.102 Safari/537.36"},
        {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_3_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.5 Safari/605.1.15"},
        {"User-Agent": "Mozilla/5.0 (Linux; Android 14; Pixel 7 Pro) Gecko/20100101 Firefox/119.0"},
        {"User-Agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Mobile Safari/537.36"}
    ]
    headers = random.choice(header_list)
    r = requests.get(url, headers=headers, timeout=20)
    soup = BeautifulSoup(r.content, 'lxml')

    brand = soup.select_one('div.-fs12.-phm.-pts a._more.-paxs.-di')
    brand = brand.text.strip() if brand else 'Brand not found'

    price = soup.select_one('div.-df.-i-ctr.-fw-w span.-fs20.-b.-ubpt.-tal.-prxs')
    price = float(re.sub(r'[^\d.]', '', price.text)) if price else None

    oldprice = soup.select_one('div.-dif.-i-ctr span.-gy5.-ubpt.-lthr.-pvxs')
    oldprice = float(re.sub(r'[^\d.]', '', oldprice.text)) if oldprice else None

    title = soup.select_one('h1.-fs16')
    title = title.text.strip() if title else 'Title not found'

    discount = soup.select_one('div.-df.-i-ctr.-fw-w span.bdg._dsct._sm.-mlxs')
    discount = discount.text.strip() if discount else 'Discount not found'

    stock = soup.select_one('p.-df.-i-ctr.-fs12.-phm.-ptxs.-yl7')
    stock = stock.text.strip() if stock else 'Stock Status not found'

    rating = soup.select_one('a.-df.-i-ctr.-pbs.-phm p.-df.-i-ctr.-ptxs span.bdg._score._sm')
    rating = rating.text.strip() if rating else 'Rating not found'

    review = soup.select_one('a.-df.-i-ctr.-pbs.-phm p.-df.-i-ctr.-ptxs span.-fs12.-mhxs')
    review = review.text.strip() if review else 'Review not found'

    date = datetime.datetime.now().strftime('%Y-%m-%d')

    return {
        'Brand': brand,
        'Title': title,
        'Price': price,
        'Old Price': oldprice,
        'Discount': discount,
        'Stock Status': stock,
        'Rating': rating,
        'Review': review,
        'URL': url,
        'Date': date
    }

In [4]:
import pandas as pd

def main():
    all_products = []

    for x in range(1, 10):
        #page_url = f'https://www.jumia.com.ng/televisions/?page={x}'
        #urls = get_url(f'https://www.jumia.com.ng/televisions/?{x}srsltid=AfmBOoop1IPRj6_et7JjxKe3nBLo7FNazzCIXj-YhBE6i8QkjU_mpJd6')
        urls = get_url(f'https://www.jumia.com.ng/televisions/?{x}srsltid=AfmBOorhF5P_SKB7GPMLqak0SuwWf8auSL9CwDfKxNQu3tzNegLA6LjW')
        products = [get_product(url) for url in urls]
        all_products.extend(products)
        print(f'Page {x} completed.')

    df = pd.DataFrame(all_products)

    df.to_csv(
        'jumia_price_history.csv',
        mode='a',
        index=False,
        header=not pd.io.common.file_exists('jumia_tv_price_history.csv')
    )

    print('Scraped and saved successfully.')

In [5]:
main()

Page 1 completed.
Page 2 completed.
Page 3 completed.
Page 4 completed.
Page 5 completed.
Page 6 completed.
Page 7 completed.
Page 8 completed.
Page 9 completed.
Scraped and saved successfully.


In [1]:
import pandas as pd
import numpy as np
import datetime

In [2]:
tv = pd.read_csv('jumia_price_history.csv')

In [4]:
tv.head(3)

,Brand,Title,Price,Old Price,Discount,Stock Status,Rating,Review,URL,Date
0,Hikers,Hikers 32'' Inches Frameless HD LED TV-Black,97999.0,124192.0,21%,Stock Status not found,4.2,169 verified ratings,https://www.jumia.com.ng/hikers-32-inches-fram...,2025-08-04
1,Hikers,Hikers 32'' Frameless Android Smart HD LED TV ...,119000.0,146784.0,19%,Stock Status not found,4.1,669 verified ratings,https://www.jumia.com.ng/hikers-32-frameless-a...,2025-08-04
2,Brand not found,Title not found,NaN,446927.0,Discount not found,Stock Status not found,Rating not found,Review not found,https://www.jumia.com.ng/50-inches-vida-smart-...,2025-08-04


In [6]:
df = tv.copy()

In [7]:
def remove_repeated_headers(df):
    header_values = [str(col).strip().lower() for col in df.columns]
    mask = df.apply(lambda row: [str(x).strip().lower() for x in row] != header_values, axis=1)
    
    return df[mask].reset_index(drop=True)

# usage
df = remove_repeated_headers(df)

In [17]:
df = df.reset_index(drop=True)

In [9]:
df['Brand'][2]

'Brand not found'

In [10]:
df = df[df['Brand'] != 'Brand not found']
df['Price'] = pd.to_numeric(df['Price'])
df['Old Price'] = pd.to_numeric(df['Old Price'])
df['Rating'] = df['Rating'].astype(str).str.replace(r'[^\d.]', '', regex=True).replace('', np.nan).astype(float)
df['Review'] = df['Review'].astype(str).str.extract(r'(\d+)').astype(float)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [24]:
df['Old Price'] = df['Old Price'].fillna(0)
df['Rating'] = df['Old Price'].fillna(0)
df['Review'] = df['Old Price'].fillna(0)

In [26]:
df.to_csv('jumia_price_history_cleaned.csv', index=False)